In [1]:
!pip install --quiet transformers torch torchvision ftfy regex tqdm packaging safetensors huggingface_hub open-clip-torch
!git clone --quiet https://github.com/UCSC-VLAA/CLIPS.git
!pip install --quiet -r CLIPS/requirements.txt

fatal: destination path 'CLIPS' already exists and is not an empty directory.


In [2]:
import torch
import torch.nn.functional as F
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration, CLIPProcessor, CLIPModel
from open_clip import create_model_from_pretrained, get_tokenizer

In [ ]:
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model     = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model.eval()

clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model     = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

clips_model, clips_preprocess = create_model_from_pretrained("hf-hub:UCSC-VLAA/ViT-L-14-CLIPS-224-Recap-DataComp-1B")
clips_tokenizer = get_tokenizer("hf-hub:UCSC-VLAA/ViT-L-14-CLIPS-224-Recap-DataComp-1B")
clips_model.eval()

image_paths = ["X:/Projects/CV/Assignment 3/Data/ILSVRC2012_test_00000003.jpg",
               "X:/Projects/CV/Assignment 3/Data/ILSVRC2012_test_00000004.jpg",
               "X:/Projects/CV/Assignment 3/Data/ILSVRC2012_test_00000018.jpg",
               "X:/Projects/CV/Assignment 3/Data/ILSVRC2012_test_00000019.jpg",
               "X:/Projects/CV/Assignment 3/Data/ILSVRC2012_test_00000022.jpg",
               "X:/Projects/CV/Assignment 3/Data/ILSVRC2012_test_00000023.jpg",
               "X:/Projects/CV/Assignment 3/Data/ILSVRC2012_test_00000025.jpg",
               "X:/Projects/CV/Assignment 3/Data/ILSVRC2012_test_00000026.jpg",
               "X:/Projects/CV/Assignment 3/Data/ILSVRC2012_test_00000030.jpg",
               "X:/Projects/CV/Assignment 3/Data/ILSVRC2012_test_00000034.jpg"]

captions = {}
for path in image_paths:
    img = Image.open(path).convert("RGB")
    inputs = blip_processor(img, return_tensors="pt")
    out = blip_model.generate(**inputs)
    caption = blip_processor.decode(out[0], skip_special_tokens=True)
    captions[path] = caption
    print(f"{path} → BLIP caption: \"{caption}\"")

print("\n=== CLIP similarity scores ===")
for path, caption in captions.items():
    img = Image.open(path).convert("RGB")
    inputs = clip_processor(text=[caption], images=img, return_tensors="pt", padding=True)
    with torch.no_grad():
        logits = clip_model(**inputs).logits_per_image
        score = logits.softmax(dim=-1)[0,0].item()
    print(f"{path}: {score:.4f}  —  \"{caption}\"")

device = torch.device("cuda")
clips_model.to(device)
print("\n=== CLIPS similarity scores ===")
for path, caption in captions.items():
    img = Image.open(path).convert("RGB")
    image_input = clips_preprocess(img).unsqueeze(0).to(device)
    text_tokens = clips_tokenizer(
        [caption], context_length=clips_model.context_length
    ).to(device)

    with torch.no_grad():
        if device.type == "cuda":
            with torch.amp.autocast(device_type="cuda"):
                img_feats = clips_model.encode_image(image_input)
                txt_feats = clips_model.encode_text(text_tokens)
        else:
            img_feats = clips_model.encode_image(image_input)
            txt_feats = clips_model.encode_text(text_tokens)

        img_feats = F.normalize(img_feats, dim=-1)
        txt_feats = F.normalize(txt_feats, dim=-1)
        logits = (100.0 * img_feats @ txt_feats.T).softmax(dim=-1)[0]
        score = logits[0].item()

    print(f"{path}: {score:.4f}  —  \"{caption}\"")

X:/Projects/CV/Assignment 3/Data/ILSVRC2012_test_00000003.jpg → BLIP caption: "a small dog walking on a green carpet"
X:/Projects/CV/Assignment 3/Data/ILSVRC2012_test_00000004.jpg → BLIP caption: "a small dog running across a green field"
X:/Projects/CV/Assignment 3/Data/ILSVRC2012_test_00000018.jpg → BLIP caption: "a family sitting in a pool with a towel"
X:/Projects/CV/Assignment 3/Data/ILSVRC2012_test_00000019.jpg → BLIP caption: "a small bird sitting on a plant"
X:/Projects/CV/Assignment 3/Data/ILSVRC2012_test_00000022.jpg → BLIP caption: "a small dog standing on a stone ledge"
X:/Projects/CV/Assignment 3/Data/ILSVRC2012_test_00000023.jpg → BLIP caption: "a man riding a bike down a wet street"
X:/Projects/CV/Assignment 3/Data/ILSVRC2012_test_00000025.jpg → BLIP caption: "a brown butterfly sitting on a green plant"
X:/Projects/CV/Assignment 3/Data/ILSVRC2012_test_00000026.jpg → BLIP caption: "a man in a suit and tie sitting on a couch"
X:/Projects/CV/Assignment 3/Data/ILSVRC2012_tes

**Metrics for CLIP–BLIP alignment**

- **Cosine similarity**  
  - Measures raw embedding alignment between image and caption.  
  - *Use when* you want a direct, scale‑agnostic score of how close the two feature vectors are.

- **Softmax probability**  
  - Converts cosine logits into a normalized “confidence” over candidates.  
  - *Use when* you need a probabilistic ranking (e.g. top‑1 accuracy) for retrieval tasks.

- **Recall@K / Top‑K accuracy**  
  - Percentage of times the correct BLIP caption appears in the top K CLIP/CLIPS matches.  
  - *Use when* evaluating multi‑candidate retrieval or caption reranking.

- **Pearson / Spearman correlation**  
  - Correlates CLIP scores with an external human‑annotated relevance scale.  
  - *Use when* assessing how well model scores align with human judgments across a dataset.

- **KL‑divergence**  
  - Quantifies the difference between CLIP vs. CLIPS softmax distributions over captions.  
  - *Use when* you want to compare the “peakiness” or spread of two models’ confidence profiles.
